# Citi Velocity swap spreads

Citi publishes a swap spread of its own. This repo also *computes* two —
`IRS_MMSS` and `IRS_SPREADOVER`, from ERIS/SDR data against UST CUSIPs — and they
are **different numbers from different inputs**. The Citi one is added alongside
rather than replacing them.

`IRS_CITIVELO_SWAP_SPREAD` is deliberately not named `..._MMSS`: `BT/signals/
tfp_swap_spread.py` selects columns with `"MMSS" in c`, so a name containing it
would have been swept into that backtest's panel silently.

**Units: basis points**, measured 2026-08-08 rather than assumed — USD_SOFR reads
2Y −14.56, 10Y −41.78, 30Y −75.11, the right magnitude, sign and term structure,
three orders of magnitude from a decimal reading.

**Against the repo's own `SPREADOVER`**, median difference over 2026-07-08..08-07:

| 2Y | 3Y | 5Y | 7Y | 10Y | 20Y | 30Y |
|---|---|---|---|---|---|---|
| +0.036 | +0.386 | +0.015 | +0.067 | −0.002 | −0.069 | −0.156 |

Under 0.4 bp on all seven, under 0.1 bp on five — two independent constructions
agreeing. Do **not** quote the mean over those days: the repo's `SPREADOVER`
fails to price on 9 of 23 days and returns values like −151,276 bp when it does,
which no mean survives.

In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import datetime
import pytz
NYC_tz = pytz.timezone("America/New_York")

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

import sys
sys.path.append("../../")

import pandas as pd
from RVUtils.plt_timeseries import make_secondary_axis_plot

In [ ]:
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB
from TB.TimeseriesBuilder import TimeseriesBuilder
from Query.Unified.UnifiedQuery import UnifiedQuery
from Query.Unified.registry import UnifiedValue

irs_mdp = IRSwapsMDP(source="CITIVELO_EXCEL-RL")
ts_builder = TimeseriesBuilder()

## The axis is ragged, so read it from the catalog

`USD_SOFR` and `USD_FEDFUND` carry **eleven** tenors — including money-market
ones, and with no 4Y/15Y/25Y. `GBP_SONIA` carries a different ten (no 1M–1Y, but
15Y/40Y/50Y). **`EUR_EUROSTR` has no `SWAP_SPREAD` node at all.** Only 13 of the
20 OIS indices carry the family, so a hardcoded tenor tuple is wrong off USD.

In [ ]:
from MDP.IRSwaps.CITIVELO_EXCEL.swap_spreads import (
    indices_with_swap_spread, swap_spread_tenors,
)

for idx in indices_with_swap_spread():
    print(f"{idx:<18} {', '.join(swap_spread_tenors(idx))}")

## Citi's published spread, per tenor

In [ ]:
start = datetime.date(2026, 7, 20)
end   = datetime.date(2026, 8, 7)

tenors = swap_spread_tenors("USD_SOFR")
queries = [
    UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_CITIVELO_SWAP_SPREAD)
    for t in tenors
]

df = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=queries,
    routers={"IRS": IRSwapsTB(irs_mdp, show_tqdm=True)},
    n_jobs=12,
    ignore_cache_miss=True,
)
df

In [ ]:
# The term structure on the last available date
last = df.dropna(how="all").iloc[-1]
last.index = [c.split()[-2] if len(c.split()) > 2 else c for c in last.index]
last.to_frame("swap spread (bp)")

## Citi's number vs the repo's computed one

`scripts/citivelo_swap_spread_tieout.py` does this properly — it excludes the days
the repo side failed to price, counts them, and persists the per-day series so the
meaningless mean cannot be quoted by accident.

In [ ]:
spreadover = [
    UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_SPREADOVER)
    for t in ("5Y", "10Y", "30Y")
]
citi = [
    UnifiedQuery(curve="USD-SOFR-1D", tenor=t, value=UnifiedValue.IRS_CITIVELO_SWAP_SPREAD)
    for t in ("5Y", "10Y", "30Y")
]

# NOTE the routers. Citi's published spread needs only the IRS router -- it is a
# quote. The repo's SPREADOVER is swap MINUS cash, so it needs an FRB router too
# and raises "IRS/FRB routers must be registered" without one. That difference IS
# the difference between the two numbers, made concrete.
from MDP.FixedRateBonds.FixedRateBondsMDP import FixedRateBondsMDP
from TB.FixedRateBondsTB import FixedRateBondsTB

usts_mdp = FixedRateBondsMDP(source="USTS_FEDINVEST_WSJ_LIVE-QL")

both = ts_builder.get_timeseries(
    start=start,
    end=end,
    queries=citi + spreadover,
    routers={
        "IRS": IRSwapsTB(irs_mdp, show_tqdm=True),
        "FRB": FixedRateBondsTB(usts_mdp, show_tqdm=True),
    },
    n_jobs=12,
    ignore_cache_miss=True,
)
both